# CCA Analysis: iCESM piControl vs. Mid-Holocene (6ka)
## Pacific SST and Southern California Precipitation

In [ ]:
import os
from importlib import reload
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy as ctp
import seaborn as sns
from sklearn import mixture, decomposition
import cartopy.crs as ccrs
import cartopy.feature as feature
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
from cartopy.util import add_cyclic_point
import cartopy.mpl.ticker as cticker
import matplotlib.ticker as mticker
from datetime import datetime, timedelta
import xesmf as xe

In [ ]:
# Load iCESM simulation data
# piControl simulations
sst_pi = xr.open_dataset("sst_pi_500.nc")
precip_pi = xr.open_dataset("precip_pi_500.nc")

# Mid-Holocene (6ka) simulations
sst_mh = xr.open_dataset("sst_mh_500.nc")
precip_mh = xr.open_dataset("precip_mh_500.nc")

print("piControl SST data:")
print(sst_pi)
print("\npiControl Precipitation data:")
print(precip_pi)
print("\n6ka SST data:")
print(sst_mh)
print("\n6ka Precipitation data:")
print(precip_mh)

In [ ]:
# Extract variable names (may vary depending on dataset)
# Common names: 'sst', 'SST', 'TS', 'PRECIP', 'PRECT', 'pr'

# Extract SST variable
if 'sst' in sst_pi.data_vars:
    sst_pi_data = sst_pi.sst
    sst_mh_data = sst_mh.sst
else:
    # Try other common names
    sst_pi_data = sst_pi[list(sst_pi.data_vars)[0]]
    sst_mh_data = sst_mh[list(sst_mh.data_vars)[0]]

# Extract precipitation variable
if 'precip' in precip_pi.data_vars:
    precip_pi_data = precip_pi.precip
    precip_mh_data = precip_mh.precip
else:
    # Try other common names
    precip_pi_data = precip_pi[list(precip_pi.data_vars)[0]]
    precip_mh_data = precip_mh[list(precip_mh.data_vars)[0]]

print(f"piControl SST variable: {sst_pi_data.name}, shape: {sst_pi_data.shape}")
print(f"piControl Precipitation variable: {precip_pi_data.name}, shape: {precip_pi_data.shape}")
print(f"6ka SST variable: {sst_mh_data.name}, shape: {sst_mh_data.shape}")
print(f"6ka Precipitation variable: {precip_mh_data.name}, shape: {precip_mh_data.shape}")

In [ ]:
def detrend_da(da, dim='time'):
    """
    Remove linear trend along dim from an xarray DataArray or Dataset variable.
    Handles NaNs by fitting only on finite values.
    """
    x = np.arange(da.sizes[dim])

    def _polyfit_nan(y):
        mask = np.isfinite(y)
        if mask.sum() < 2:
            return np.array([np.nan, np.nan], dtype=float)
        return np.polyfit(x[mask], y[mask], 1)

    coeffs = xr.apply_ufunc(
        _polyfit_nan,
        da,
        input_core_dims=[[dim]],
        output_core_dims=[["coef"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
    )

    slope = coeffs.sel(coef=0)
    intercept = coeffs.sel(coef=1)
    x_da = xr.DataArray(x, dims=[dim], coords={dim: da[dim]})
    trend = slope * x_da + intercept
    return da - trend

# Detrend the data
print("Detrending piControl SST...")
sst_pi_detrended = detrend_da(sst_pi_data, dim='time')
print("Detrending piControl Precipitation...")
precip_pi_detrended = detrend_da(precip_pi_data, dim='time')
print("Detrending 6ka SST...")
sst_mh_detrended = detrend_da(sst_mh_data, dim='time')
print("Detrending 6ka Precipitation...")
precip_mh_detrended = detrend_da(precip_mh_data, dim='time')
print("Detrending complete!")

## piControl Simulation Analysis

In [ ]:
# Subset piControl SST for extended winter (Nov-Apr) in the region 100°E-100°W, 26°S-66°N (tropical and North Pacific)
sst_pi_subset = sst_pi_detrended.sel(
    lon=slice(100, 260),  # 100°E to 100°W (100 to 260 in 0-360 convention)
    lat=slice(66, -26)
)
sst_pi_subset = sst_pi_subset.isel(time=sst_pi_subset.time.dt.month.isin([11, 12, 1, 2, 3, 4]))

# Subset piControl precipitation for extended winter (Nov-Apr) in Southern California (122°W-114°W, 32°N-36°N)
precip_pi_subset = precip_pi_detrended.sel(
    lon=slice(-122, -114),
    lat=slice(32, 36)
)
precip_pi_subset = precip_pi_subset.isel(time=precip_pi_subset.time.dt.month.isin([11, 12, 1, 2, 3, 4]))

print("piControl SST subset shape:", sst_pi_subset.shape)
print("piControl Precipitation subset shape:", precip_pi_subset.shape)

In [ ]:
# Group by year and calculate extended winter averages for piControl
# Extended winter: Nov of previous year + Dec, Jan, Feb, Mar, Apr of current year

# For SST subset
sst_pi_season_year = sst_pi_subset.time.dt.year.copy()
# Shift November dates to the following year for grouping
sst_pi_season_year = sst_pi_season_year.where(sst_pi_subset.time.dt.month != 11, sst_pi_subset.time.dt.year + 1)

sst_pi_seasonal_mean = sst_pi_subset.groupby(sst_pi_season_year).mean(dim='time')
sst_pi_seasonal_mean = sst_pi_seasonal_mean.rename({'year': 'season_year'})

# For precipitation subset
precip_pi_season_year = precip_pi_subset.time.dt.year.copy()
# Shift November dates to the following year for grouping
precip_pi_season_year = precip_pi_season_year.where(precip_pi_subset.time.dt.month != 11, precip_pi_subset.time.dt.year + 1)

precip_pi_seasonal_mean = precip_pi_subset.groupby(precip_pi_season_year).mean(dim='time')
precip_pi_seasonal_mean = precip_pi_seasonal_mean.rename({'year': 'season_year'})

print("piControl SST seasonal mean shape:", sst_pi_seasonal_mean.shape)
print("piControl Precipitation seasonal mean shape:", precip_pi_seasonal_mean.shape)

In [ ]:
# PCA for piControl data
# Reshape SST and precipitation data for PCA
# Flatten spatial dimensions while keeping time dimension

# piControl SST: reshape to (time, space)
sst_pi_data_reshaped = sst_pi_seasonal_mean.values.reshape(sst_pi_seasonal_mean.sizes['season_year'], -1)

# piControl Precipitation: reshape to (time, space)
precip_pi_data_reshaped = precip_pi_seasonal_mean.values.reshape(precip_pi_seasonal_mean.sizes['season_year'], -1)

# Remove NaN values for PCA
sst_pi_valid = sst_pi_data_reshaped[:, ~np.isnan(sst_pi_data_reshaped[0])]
precip_pi_valid = precip_pi_data_reshaped[:, ~np.isnan(precip_pi_data_reshaped[0])]

# Apply PCA to extract leading components
n_components_sst = 10
n_components_precip = 5

pca_sst_pi = decomposition.PCA(n_components=n_components_sst)
pca_precip_pi = decomposition.PCA(n_components=n_components_precip)

# Fit PCA and transform data
sst_pi_pca = pca_sst_pi.fit_transform(sst_pi_valid)
precip_pi_pca = pca_precip_pi.fit_transform(precip_pi_valid)

# Print explained variance ratio
print(f"piControl SST - Explained variance ratio: {pca_sst_pi.explained_variance_ratio_}")
print(f"piControl SST - Cumulative explained variance: {np.cumsum(pca_sst_pi.explained_variance_ratio_)}")
print(f"\npiControl Precipitation - Explained variance ratio: {pca_precip_pi.explained_variance_ratio_}")
print(f"piControl Precipitation - Cumulative explained variance: {np.cumsum(pca_precip_pi.explained_variance_ratio_)}")

In [ ]:
# Build PCA-score DataArrays with real time coordinates for piControl
sst_pi_pca_scores = xr.DataArray(
    sst_pi_pca,
    dims=("season_year", "sst_component"),
    coords={
        "season_year": sst_pi_seasonal_mean.season_year.values,
        "sst_component": np.arange(1, n_components_sst + 1),
    },
)

precip_pi_pca_scores = xr.DataArray(
    precip_pi_pca,
    dims=("season_year", "precip_component"),
    coords={
        "season_year": precip_pi_seasonal_mean.season_year.values,
        "precip_component": np.arange(1, n_components_precip + 1),
    },
)

# Keep only the overlapping months between the two PCA score series
sst_pi_aligned, precip_pi_aligned = xr.align(
    sst_pi_pca_scores, precip_pi_pca_scores, join="inner"
)

print(f"piControl SST aligned shape: {sst_pi_aligned.data.shape}")
print(f"piControl Precipitation aligned shape: {precip_pi_aligned.data.shape}")

In [ ]:
from sklearn.cross_decomposition import CCA

# Perform CCA for piControl
cca_pi = CCA(n_components=3)
cca_pi.fit(sst_pi_aligned.data, precip_pi_aligned.data)

# Transform the data to get the canonical variates
sst_pi_c, precip_pi_c = cca_pi.transform(sst_pi_aligned.data, precip_pi_aligned.data)

print("piControl CCA Complete")
print(f"piControl SST canonical variates shape: {sst_pi_c.shape}")
print(f"piControl Precipitation canonical variates shape: {precip_pi_c.shape}")

In [ ]:
# Calculate correlations for piControl
correlations_pi = []
print("piControl Canonical Correlations:")
for i in range(3):
    corr = np.corrcoef(sst_pi_c[:, i], precip_pi_c[:, i])[0, 1]
    correlations_pi.append(corr)
    print(f"  CC{i+1}: {corr:.4f}")

print(f"\npiControl correlations: {correlations_pi}")

## 6ka (Mid-Holocene) Simulation Analysis

In [ ]:
# Subset 6ka SST for extended winter (Nov-Apr) in the region 100°E-100°W, 26°S-66°N
sst_mh_subset = sst_mh_detrended.sel(
    lon=slice(100, 260),  # 100°E to 100°W (100 to 260 in 0-360 convention)
    lat=slice(66, -26)
)
sst_mh_subset = sst_mh_subset.isel(time=sst_mh_subset.time.dt.month.isin([11, 12, 1, 2, 3, 4]))

# Subset 6ka precipitation for extended winter (Nov-Apr) in Southern California
precip_mh_subset = precip_mh_detrended.sel(
    lon=slice(-122, -114),
    lat=slice(32, 36)
)
precip_mh_subset = precip_mh_subset.isel(time=precip_mh_subset.time.dt.month.isin([11, 12, 1, 2, 3, 4]))

print("6ka SST subset shape:", sst_mh_subset.shape)
print("6ka Precipitation subset shape:", precip_mh_subset.shape)

In [ ]:
# Group by year and calculate extended winter averages for 6ka
# Extended winter: Nov of previous year + Dec, Jan, Feb, Mar, Apr of current year

# For SST subset
sst_mh_season_year = sst_mh_subset.time.dt.year.copy()
# Shift November dates to the following year for grouping
sst_mh_season_year = sst_mh_season_year.where(sst_mh_subset.time.dt.month != 11, sst_mh_subset.time.dt.year + 1)

sst_mh_seasonal_mean = sst_mh_subset.groupby(sst_mh_season_year).mean(dim='time')
sst_mh_seasonal_mean = sst_mh_seasonal_mean.rename({'year': 'season_year'})

# For precipitation subset
precip_mh_season_year = precip_mh_subset.time.dt.year.copy()
# Shift November dates to the following year for grouping
precip_mh_season_year = precip_mh_season_year.where(precip_mh_subset.time.dt.month != 11, precip_mh_subset.time.dt.year + 1)

precip_mh_seasonal_mean = precip_mh_subset.groupby(precip_mh_season_year).mean(dim='time')
precip_mh_seasonal_mean = precip_mh_seasonal_mean.rename({'year': 'season_year'})

print("6ka SST seasonal mean shape:", sst_mh_seasonal_mean.shape)
print("6ka Precipitation seasonal mean shape:", precip_mh_seasonal_mean.shape)

In [ ]:
# PCA for 6ka data
# Reshape SST and precipitation data for PCA

# 6ka SST: reshape to (time, space)
sst_mh_data_reshaped = sst_mh_seasonal_mean.values.reshape(sst_mh_seasonal_mean.sizes['season_year'], -1)

# 6ka Precipitation: reshape to (time, space)
precip_mh_data_reshaped = precip_mh_seasonal_mean.values.reshape(precip_mh_seasonal_mean.sizes['season_year'], -1)

# Remove NaN values for PCA
sst_mh_valid = sst_mh_data_reshaped[:, ~np.isnan(sst_mh_data_reshaped[0])]
precip_mh_valid = precip_mh_data_reshaped[:, ~np.isnan(precip_mh_data_reshaped[0])]

# Apply PCA to extract leading components
pca_sst_mh = decomposition.PCA(n_components=n_components_sst)
pca_precip_mh = decomposition.PCA(n_components=n_components_precip)

# Fit PCA and transform data
sst_mh_pca = pca_sst_mh.fit_transform(sst_mh_valid)
precip_mh_pca = pca_precip_mh.fit_transform(precip_mh_valid)

# Print explained variance ratio
print(f"6ka SST - Explained variance ratio: {pca_sst_mh.explained_variance_ratio_}")
print(f"6ka SST - Cumulative explained variance: {np.cumsum(pca_sst_mh.explained_variance_ratio_)}")
print(f"\n6ka Precipitation - Explained variance ratio: {pca_precip_mh.explained_variance_ratio_}")
print(f"6ka Precipitation - Cumulative explained variance: {np.cumsum(pca_precip_mh.explained_variance_ratio_)}")

In [ ]:
# Build PCA-score DataArrays with real time coordinates for 6ka
sst_mh_pca_scores = xr.DataArray(
    sst_mh_pca,
    dims=("season_year", "sst_component"),
    coords={
        "season_year": sst_mh_seasonal_mean.season_year.values,
        "sst_component": np.arange(1, n_components_sst + 1),
    },
)

precip_mh_pca_scores = xr.DataArray(
    precip_mh_pca,
    dims=("season_year", "precip_component"),
    coords={
        "season_year": precip_mh_seasonal_mean.season_year.values,
        "precip_component": np.arange(1, n_components_precip + 1),
    },
)

# Keep only the overlapping months between the two PCA score series
sst_mh_aligned, precip_mh_aligned = xr.align(
    sst_mh_pca_scores, precip_mh_pca_scores, join="inner"
)

print(f"6ka SST aligned shape: {sst_mh_aligned.data.shape}")
print(f"6ka Precipitation aligned shape: {precip_mh_aligned.data.shape}")

In [ ]:
# Perform CCA for 6ka
cca_mh = CCA(n_components=3)
cca_mh.fit(sst_mh_aligned.data, precip_mh_aligned.data)

# Transform the data to get the canonical variates
sst_mh_c, precip_mh_c = cca_mh.transform(sst_mh_aligned.data, precip_mh_aligned.data)

print("6ka CCA Complete")
print(f"6ka SST canonical variates shape: {sst_mh_c.shape}")
print(f"6ka Precipitation canonical variates shape: {precip_mh_c.shape}")

In [ ]:
# Calculate correlations for 6ka
correlations_mh = []
print("6ka Canonical Correlations:")
for i in range(3):
    corr = np.corrcoef(sst_mh_c[:, i], precip_mh_c[:, i])[0, 1]
    correlations_mh.append(corr)
    print(f"  CC{i+1}: {corr:.4f}")

print(f"\n6ka correlations: {correlations_mh}")

## Comparison: piControl vs. 6ka

In [ ]:
# Compare canonical correlations between piControl and 6ka
print("="*60)
print("CANONICAL CORRELATION COMPARISON: piControl vs. 6ka")
print("="*60)
print(f"\n{'Mode':<10} {'piControl':<15} {'6ka':<15} {'Difference':<15}")
print("-"*60)

for i in range(3):
    diff = correlations_pi[i] - correlations_mh[i]
    print(f"CC{i+1:<8} {correlations_pi[i]:<15.4f} {correlations_mh[i]:<15.4f} {diff:<15.4f}")

print("\n" + "="*60)

In [ ]:
# Create comparison plots of canonical correlations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

modes = ['CC1', 'CC2', 'CC3']
x = np.arange(len(modes))
width = 0.35

# Plot 1: Bar chart of correlations comparison
ax = axes[0, 0]
ax.bar(x - width/2, correlations_pi, width, label='piControl', alpha=0.8)
ax.bar(x + width/2, correlations_mh, width, label='6ka', alpha=0.8)
ax.set_xlabel('Canonical Mode', fontsize=11)
ax.set_ylabel('Canonical Correlation', fontsize=11)
ax.set_title('Canonical Correlations: piControl vs. 6ka', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(modes)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='y')

# Plot 2: Differences
ax = axes[0, 1]
differences = [correlations_pi[i] - correlations_mh[i] for i in range(3)]
colors = ['green' if d > 0 else 'red' for d in differences]
ax.bar(modes, differences, color=colors, alpha=0.7)
ax.set_xlabel('Canonical Mode', fontsize=11)
ax.set_ylabel('Correlation Difference (piControl - 6ka)', fontsize=11)
ax.set_title('Difference in Canonical Correlations', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax.grid(alpha=0.3, axis='y')

# Plot 3: piControl time series
ax = axes[1, 0]
years_pi = sst_pi_aligned.season_year.values
ax.plot(years_pi, sst_pi_c[:, 0], 'r-', label='SST', alpha=0.7, linewidth=1.5)
ax.plot(years_pi, precip_pi_c[:, 0], 'b-', label='Precip', alpha=0.7, linewidth=1.5)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Canonical Variate (CC1)', fontsize=11)
ax.set_title(f'piControl CC1 Time Series (r={correlations_pi[0]:.3f})', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Plot 4: 6ka time series
ax = axes[1, 1]
years_mh = sst_mh_aligned.season_year.values
ax.plot(years_mh, sst_mh_c[:, 0], 'r-', label='SST', alpha=0.7, linewidth=1.5)
ax.plot(years_mh, precip_mh_c[:, 0], 'b-', label='Precip', alpha=0.7, linewidth=1.5)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Canonical Variate (CC1)', fontsize=11)
ax.set_title(f'6ka CC1 Time Series (r={correlations_mh[0]:.3f})', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('CCA_piControl_vs_6ka_comparison.png', dpi=300, bbox_inches='tight')
print("Comparison plot saved as 'CCA_piControl_vs_6ka_comparison.png'")
plt.show()

In [ ]:
# Summary statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

print("\npiControl:")
print(f"  Number of seasons: {len(sst_pi_aligned.season_year)}")
print(f"  Mean CC1 correlation: {correlations_pi[0]:.4f}")
print(f"  Mean CC2 correlation: {correlations_pi[1]:.4f}")
print(f"  Mean CC3 correlation: {correlations_pi[2]:.4f}")
print(f"  Average of all CCs: {np.mean(correlations_pi):.4f}")

print("\n6ka (Mid-Holocene):")
print(f"  Number of seasons: {len(sst_mh_aligned.season_year)}")
print(f"  Mean CC1 correlation: {correlations_mh[0]:.4f}")
print(f"  Mean CC2 correlation: {correlations_mh[1]:.4f}")
print(f"  Mean CC3 correlation: {correlations_mh[2]:.4f}")
print(f"  Average of all CCs: {np.mean(correlations_mh):.4f}")

print("\nDifferences (piControl - 6ka):")
print(f"  CC1 difference: {correlations_pi[0] - correlations_mh[0]:+.4f}")
print(f"  CC2 difference: {correlations_pi[1] - correlations_mh[1]:+.4f}")
print(f"  CC3 difference: {correlations_pi[2] - correlations_mh[2]:+.4f}")
print(f"  Average difference: {np.mean(correlations_pi) - np.mean(correlations_mh):+.4f}")

print("\n" + "="*70)

In [ ]:
print("\nCCA Analysis Complete!")
print("\nThis analysis compared:")
print("  - Pacific SST (tropical and North Pacific, 100°E-100°W, 26°S-66°N)")
print("  - Southern California Precipitation (122°W-114°W, 32°N-36°N)")
print("  - Extended winter season (Nov-Apr)")
print("\nBetween:")
print("  - piControl simulation (pre-industrial control)")
print("  - 6ka simulation (Mid-Holocene, 6000 years ago)")